# Distance Metrics — Hands-on Tutorial

In this notebook we'll build intuition for how different distance metrics work:
1. **L1 and L2 distances** — interactive explorer with sliders
2. **Unit circles** — what "distance = 1" looks like under different metrics
3. **Cosine distance** — a gotcha about magnitude
4. **Mahalanobis distance** — why data shape matters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from scipy.spatial.distance import mahalanobis

---
## Part 1: Interactive Distance Explorer

Use the sliders to move two points around the 2D plane.

The plot shows:
- The **L1 (Manhattan) path** — the stepped route (move horizontally, then vertically)
- The **L2 (Euclidean) path** — the straight line

The three distance values update live:
- $d_{L1}(p, q) = |x_1 - x_2| + |y_1 - y_2|$
- $d_{L2}(p, q) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$
- $d_{\cos}(p, q) = 1 - \frac{p \cdot q}{\|p\| \|q\|}$

In [ ]:
def plot_distances(x1, y1, x2, y2):
    """Plot two points with L1 stepped path and L2 straight line."""
    p = np.array([x1, y1])
    q = np.array([x2, y2])

    # Compute distances
    d_l1 = np.abs(x1 - x2) + np.abs(y1 - y2)
    d_l2 = np.linalg.norm(p - q)

    # Cosine distance (handle zero vectors)
    norm_p, norm_q = np.linalg.norm(p), np.linalg.norm(q)
    if norm_p > 0 and norm_q > 0:
        d_cos = 1 - np.dot(p, q) / (norm_p * norm_q)
    else:
        d_cos = float('nan')

    fig, ax = plt.subplots(figsize=(7, 7))

    # L1 stepped path: go horizontal first, then vertical
    ax.plot([x1, x2, x2], [y1, y1, y2], color='tab:orange', linewidth=2,
            linestyle='--', label=f'L1 = {d_l1:.2f}', zorder=2)

    # L2 straight line
    ax.plot([x1, x2], [y1, y2], color='tab:blue', linewidth=2,
            label=f'L2 = {d_l2:.2f}', zorder=2)

    # Points
    ax.scatter(*p, color='red', s=100, zorder=3)
    ax.scatter(*q, color='green', s=100, zorder=3)
    ax.annotate(f'P ({x1:.1f}, {y1:.1f})', p, textcoords="offset points",
                xytext=(10, 10), fontsize=10)
    ax.annotate(f'Q ({x2:.1f}, {y2:.1f})', q, textcoords="offset points",
                xytext=(10, 10), fontsize=10)

    ax.set_xlim(-5.5, 5.5)
    ax.set_ylim(-5.5, 5.5)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(f'Cosine distance = {d_cos:.4f}')
    ax.legend(loc='upper left', fontsize=11)
    plt.tight_layout()
    plt.show()

slider_kw = dict(min=-5.0, max=5.0, step=0.1)
interact(
    plot_distances,
    x1=FloatSlider(value=1.0, description='x₁', **slider_kw),
    y1=FloatSlider(value=2.0, description='y₁', **slider_kw),
    x2=FloatSlider(value=4.0, description='x₂', **slider_kw),
    y2=FloatSlider(value=-1.0, description='y₂', **slider_kw),
);

### Think about it

- Move the points so that L1 and L2 are equal. When does that happen?
- Move the points so that L1 is *much* larger than L2. What configuration causes that?
- Can L2 ever be larger than L1?
- What happens to the cosine distance when one of the two points is on (0,0)?

---
## Part 2: Unit Circles Under Different Metrics

A "unit circle" is the set of all points at distance exactly 1 from the origin.
The *shape* of that set depends on which distance metric you use:

| Metric | Formula | Shape |
|--------|---------|-------|
| L1 (Manhattan) | $\|x\| + \|y\| = 1$ | Diamond |
| L2 (Euclidean) | $\sqrt{x^2 + y^2} = 1$ | Circle |

In [ ]:
theta = np.linspace(0, 2 * np.pi, 500)

# L2 unit circle — the familiar one
x_l2 = np.cos(theta)
y_l2 = np.sin(theta)

# L1 unit circle — diamond: parameterize via |x| + |y| = 1
# Use the L2 circle and normalize by L1 norm
l1_norm = np.abs(np.cos(theta)) + np.abs(np.sin(theta))
x_l1 = np.cos(theta) / l1_norm
y_l1 = np.sin(theta) / l1_norm

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(x_l1, y_l1, label='L1 (diamond)', linewidth=2)
ax.plot(x_l2, y_l2, label='L2 (circle)', linewidth=2)

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='grey', linewidth=0.5)
ax.axvline(0, color='grey', linewidth=0.5)
ax.set_title('Unit circles under L1 and L2 metrics')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### Think about it

- Why is the L1 "circle" a diamond? Think about which points satisfy $|x| + |y| = 1$.
- The L1 diamond touches the axes at $(\pm 1, 0)$ and $(0, \pm 1)$, the same places as L2. Where do they differ most?
- In which metric is the point $(0.7, 0.7)$ *outside* the unit circle? Inside?

---
## Part 3: Cosine Distance Gotcha

Cosine distance measures the *angle* between two vectors — it completely ignores magnitude.

$$d_{\cos}(p, q) = 1 - \frac{p \cdot q}{\|p\|\;\|q\|}$$

This means two vectors pointing in the same direction have cosine distance **0**,
no matter how far apart they are in Euclidean space.

In [ ]:
# Three vectors: same direction, different magnitudes
vectors = {
    'A = (1, 1)':    np.array([1, 1]),
    'B = (5, 5)':    np.array([5, 5]),
    'C = (10, 10)':  np.array([10, 10]),
}

# Compute pairwise distances
print(f"{'Pair':<20} {'L2 distance':>12} {'Cosine distance':>16}")
print('-' * 50)

names = list(vectors.keys())
vecs = list(vectors.values())
for i in range(len(vecs)):
    for j in range(i + 1, len(vecs)):
        d_l2 = np.linalg.norm(vecs[i] - vecs[j])
        cos_sim = np.dot(vecs[i], vecs[j]) / (np.linalg.norm(vecs[i]) * np.linalg.norm(vecs[j]))
        d_cos = 1 - cos_sim
        print(f"{names[i]:>8} vs {names[j]:<8} {d_l2:>12.4f} {d_cos:>16.6f}")

In [ ]:
# Visualize: vectors of different lengths but same (or different) directions
fig, ax = plt.subplots(figsize=(7, 7))

colors = ['tab:blue', 'tab:orange', 'tab:green']
for (name, vec), color in zip(vectors.items(), colors):
    ax.annotate('', xy=vec, xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    ax.annotate(name, xy=vec, textcoords="offset points",
                xytext=(10, 5), fontsize=10, color=color)

# Add a vector in a different direction for contrast
d = np.array([2, -1])
ax.annotate('', xy=d, xytext=(0, 0),
            arrowprops=dict(arrowstyle='->', color='tab:red', lw=2.5))
ax.annotate(f'D = ({d[0]}, {d[1]})', xy=d, textcoords="offset points",
            xytext=(10, -15), fontsize=10, color='tab:red')

# Print cosine distance A vs D
cos_ad = 1 - np.dot(vecs[0], d) / (np.linalg.norm(vecs[0]) * np.linalg.norm(d))
ax.set_title(f'Cosine dist(A, B) = 0.0  |  Cosine dist(A, D) = {cos_ad:.4f}')

ax.set_xlim(-1, 12)
ax.set_ylim(-3, 12)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='grey', linewidth=0.5)
ax.axvline(0, color='grey', linewidth=0.5)
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.tight_layout()
plt.show()

### Think about it

- A, B, and C all point in the same direction. Cosine says they're identical — L2 says they're far apart. Which metric is "right"?
- **When cosine makes sense:** comparing documents by word frequency (a longer document isn't necessarily *different*, just longer). Or comparing user preference vectors.
- **When cosine is misleading:** if magnitude carries real information (e.g., sensor readings, physical measurements), ignoring it loses signal.

---
## Part 4: Mahalanobis Distance

Euclidean distance treats all directions equally. But if your data is stretched or tilted
(i.e., the features are correlated), a point that *looks* close in L2 might actually be
an outlier — and vice versa.

**Mahalanobis distance** accounts for the shape (covariance) of the data:

$$d_M(x, \mu) = \sqrt{(x - \mu)^T \, \Sigma^{-1} \, (x - \mu)}$$

where $\Sigma$ is the covariance matrix and $\mu$ is the mean.

Intuitively: it measures distance in units of "how many standard deviations away",
stretched along the principal axes of the data.

In [ ]:
# Generate a tilted elliptical Gaussian cluster
np.random.seed(7)

# Covariance matrix: stretched along a diagonal axis
cov = np.array([[3.0, 2.0],
                [2.0, 2.0]])
mean = np.array([0.0, 0.0])
data = np.random.multivariate_normal(mean, cov, size=300)

# Two test points
# Point P: close in L2 but perpendicular to the covariance axis (outlier)
point_p = np.array([1.0, -2.0])

# Point Q: farther in L2 but along the covariance axis (inlier)
point_q = np.array([3.0, 2.5])

# Compute distances from the mean
cov_inv = np.linalg.inv(cov)

d_l2_p = np.linalg.norm(point_p - mean)
d_mah_p = mahalanobis(point_p, mean, cov_inv)

d_l2_q = np.linalg.norm(point_q - mean)
d_mah_q = mahalanobis(point_q, mean, cov_inv)

print(f"Point P = {point_p}")
print(f"  L2 distance from mean:          {d_l2_p:.3f}")
print(f"  Mahalanobis distance from mean:  {d_mah_p:.3f}")
print()
print(f"Point Q = {point_q}")
print(f"  L2 distance from mean:          {d_l2_q:.3f}")
print(f"  Mahalanobis distance from mean:  {d_mah_q:.3f}")
print()
print("P is closer in L2, but Q is closer in Mahalanobis!")

In [ ]:
# Visualize the data, covariance ellipses, and test points
from matplotlib.patches import Ellipse

def plot_cov_ellipse(ax, mean, cov, n_std, **kwargs):
    """Plot an ellipse at n_std standard deviations from the mean."""
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    # eigh returns eigenvalues ascending; index 1 is the major axis
    angle = np.degrees(np.arctan2(eigenvectors[1, 1], eigenvectors[0, 1]))
    # width = major axis (larger eigenvalue), height = minor axis
    width = 2 * n_std * np.sqrt(eigenvalues[1])
    height = 2 * n_std * np.sqrt(eigenvalues[0])
    ellipse = Ellipse(xy=mean, width=width, height=height, angle=angle, **kwargs)
    ax.add_patch(ellipse)

fig, ax = plt.subplots(figsize=(8, 8))

# Data cloud
ax.scatter(data[:, 0], data[:, 1], s=8, alpha=0.4, color='grey', label='Data')

# Covariance ellipses at 1, 2, 3 sigma
for n_std in [1, 2, 3]:
    plot_cov_ellipse(ax, mean, cov, n_std,
                     fill=False, edgecolor='steelblue', linewidth=1.5, linestyle='--')

# L2 circle at same radius as point P's L2 distance (for reference)
circle = plt.Circle(mean, d_l2_p, fill=False, color='grey', linestyle=':', linewidth=1,
                    label=f'L2 circle (r={d_l2_p:.2f})')
ax.add_patch(circle)

# Test points
ax.scatter(*point_p, color='red', s=150, zorder=5, edgecolors='black', linewidths=1.5)
ax.annotate(f'P  (L2={d_l2_p:.2f}, Mah={d_mah_p:.2f})',
            point_p, textcoords="offset points", xytext=(12, -12), fontsize=10,
            color='red', fontweight='bold')

ax.scatter(*point_q, color='green', s=150, zorder=5, edgecolors='black', linewidths=1.5)
ax.annotate(f'Q  (L2={d_l2_q:.2f}, Mah={d_mah_q:.2f})',
            point_q, textcoords="offset points", xytext=(12, -12), fontsize=10,
            color='green', fontweight='bold')
# Mark the mean
ax.scatter(*mean, color='black', s=80, marker='+', zorder=5, linewidths=2)

ax.set_xlim(-6, 7)
ax.set_ylim(-6, 7)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Mahalanobis vs Euclidean: data shape matters')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

### Think about it

- **Red point P** is closer to the mean in L2, but it falls *off* the data's main axis — it's an outlier by Mahalanobis.
- **Green point Q** is farther in L2, but it lies *along* the direction the data naturally varies — it's an inlier by Mahalanobis.
- The grey dotted circle shows where L2 draws the boundary. The blue dashed ellipses show where Mahalanobis draws it. Which one better captures "unusual"?
- **Connection to anomaly detection:** if you threshold on L2, you'd flag Q but miss P. Mahalanobis correctly identifies P as the outlier.